# 00. Thinking 模式与 Test-time Compute

模型权重训练完就固定了，但还有一个维度可以提升能力：**推理时花更多 token 思考**。
Thinking 模式让模型先输出推理链（reasoning chain），再给最终答案 — 多步推理题的正确率显著提升，
代价是 completion tokens 成倍增长。

实践原则：**按问题配模式**。简单回忆题开 thinking 是纯浪费；多步推理不开 thinking 是放弃正确率。

> **检查点**：能说出 thinking 模式的成本公式：成本 ≈ reasoning tokens + answer tokens，且 reasoning 部分通常占 80%+。


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
from pathlib import Path
import subprocess, sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent

SCRIPT = ROOT / 'agents' / 'deep-research' / 'scripts' / 'thinking_compare.py'
result = subprocess.run([sys.executable, str(SCRIPT)], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)


Test-time compute: trade more inference tokens for higher accuracy.
Thinking mode makes the model emit a reasoning chain before the final answer.

Problems used in this demo:
  - A farmer has 17 sheep. All but 9 run away. How many are left? Answer with just the number.  (expect: 9)
  - If a bat and a ball cost $1.10 total and the bat costs $1.00 more than the ball, how many cents does the ball cost? Answer with just the number.  (expect: 5)
  - Alice is taller than Bob. Bob is taller than Carol. David is shorter than Carol. Who is the shortest? Answer with just the name.  (expect: David)

Run with --use-api to see the real comparison.
Toy mode cannot simulate reasoning — that's the whole point of thinking mode.



## 什么时候开 thinking

| 问题类型 | 建议 | 原因 |
|---------|------|------|
| 事实回忆（「Python 哪年发布」） | 关 | 一步直达，推理链纯浪费 |
| 多步数学/逻辑 | 开 | 中间步骤显著降低错误率 |
| 代码调试 | 开 | 需要假设-验证循环 |
| 创意写作 | 关 | 无标准答案，推理链反而限制发散 |
| Agent 规划步 | 按步数 | 规划深时开，执行步通常关 |

Reasoning effort（low/medium/high）是中间档：用 effort 控制「想多久」。


In [3]:
from pathlib import Path
import subprocess, sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent

SCRIPT = ROOT / 'agents' / 'deep-research' / 'scripts' / 'thinking_compare.py'
result = subprocess.run([sys.executable, str(SCRIPT), '--use-api', '--verbose'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)


Model: deepseek-v4-pro  effort: default

=== Problem 1: A farmer has 17 sheep. All but 9 run away. How many are left... (expect: 9) ===
  thinking=off: answer='9' correct=True tokens=58 latency=1.8s
    reasoning preview: We are asked: "A farmer has 17 sheep. All but 9 run away. How many are left?" The phrase "All but 9 run away" means that all sheep except 9 ran away, so 9 sheep remain. So answer is 9....
  thinking= on: answer='9' correct=True tokens=63 latency=1.3s
    reasoning preview: We are given a simple riddle: "A farmer has 17 sheep. All but 9 run away. How many are left?" The phrase "all but 9 run away" means that all sheep except 9 ran away, so 9 sheep are left. The answer is...

=== Problem 2: If a bat and a ball cost $1.10 total and the bat costs $1.00... (expect: 5) ===
  thinking=off: answer='5' correct=True tokens=138 latency=2.1s
    reasoning preview: We are asked: "If a bat and a ball cost $1.10 total and the bat costs $1.00 more than the ball, how many cents does t

## 真实对比观察

看上面的 Summary：thinking=on 的 completion tokens 通常是 off 的 5-20 倍。
注意 bat-and-ball 这类「经典陷阱题」：不开 thinking 时模型倾向脱口而出 10 美分。

## 练习

1. 用 `--effort low` 和 `--effort high` 各跑一次，绘制 effort × 正确率 × token 的三角权衡。
2. 自己加一道你领域的多步推理题到 `PROBLEMS`，验证 thinking 的收益。
3. 思考：agent 循环中哪些「步」值得开 thinking？（提示：规划/反思步 vs 工具参数填充步）
